In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import scanpy as sc
import episcanpy as epi
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

import glob
import sklearn
import pyreadr
import scipy
import os

from pathlib import Path
from helps import *

# Raw CM

## Peak

In [ ]:
atac=sc.read_mtx("Datasets/MouseSkin/GSM4156597_skin.late.anagen.counts.txt.gz").T
atac.X=scipy.sparse.csr_matrix(atac.X, dtype="int32")
atac.layers["raw"]=atac.X.copy()
atac.var=pd.read_csv("Datasets/MouseSkin/GSM4156597_skin.late.anagen.peaks.bed.gz", sep="\t", header=None)
atac

In [ ]:
atac.obs.index=pd.read_csv("Datasets/MouseSkin/GSM4156597_skin.late.anagen.barcodes.txt.gz", sep="\t", header=None)[0]
atac.obs.index.name=""
atac.obs.head()

In [ ]:
metadata=pd.read_csv("Datasets/MouseSkin/GSM4156597_skin_celltype.txt.gz", sep="\t")
print(metadata.shape)
metadata.index=metadata["rna.bc"]
metadata.head()

In [ ]:
atac.obs=metadata.loc[atac.obs.index]
atac.obs.index=atac.obs["atac.bc"]
atac.obs.index.name="barcode"
atac.obs.head()

In [ ]:
atac.var.index=atac.var[[0, 1, 2]].astype(str).agg('_'.join, axis=1)
atac.var.rename({0 : "chr", 1 : "start", 2 : "stop"}, axis=1, inplace=True)
atac.var.head()

In [ ]:
atac.obs["atac.bc"]=atac.obs["atac.bc"].str.replace(".", "_")
atac.obs["atac.bc"]=atac.obs["atac.bc"].str.replace(",", "_")
atac.obs["rna.bc"]=atac.obs["rna.bc"].str.replace(".", "_")
atac.obs["rna.bc"]=atac.obs["rna.bc"].str.replace(",", "_")
atac.obs.index=atac.obs.index.str.replace(".", "_")
atac.obs.index=atac.obs.index.str.replace(",", "_")
atac.obs

In [ ]:
atac.obs.rename({"celltype" : "CellType"}, axis=1, inplace=True)

In [ ]:
Path("Datasets/MouseSkin/CM").mkdir(parents=True, exist_ok=True)
atac.write("Datasets/MouseSkin/CM/MouseSkin_Peak_Raw.h5ad", compression="gzip")

## GEX

In [ ]:
rna=sc.read("Datasets/MouseSkin/GSM4156608_skin.late.anagen.rna.counts.txt.gz").T
rna.X=scipy.sparse.csr_matrix(rna.X, dtype="int32")
rna.layers["raw"]=rna.X.copy()
rna

In [ ]:
rna.obs.index=rna.obs.index.str.replace(".", "_")
rna.obs.index=rna.obs.index.str.replace(",", "_")
rna.obs.head()

In [ ]:
metadata=pd.read_csv("Datasets/MouseSkin/GSM4156597_skin_celltype.txt.gz", sep="\t")
print(metadata.shape)
metadata["atac.bc"]=metadata["atac.bc"].str.replace(".", "_")
metadata["atac.bc"]=metadata["atac.bc"].str.replace(",", "_")
metadata["rna.bc"]=metadata["rna.bc"].str.replace(".", "_")
metadata["rna.bc"]=metadata["rna.bc"].str.replace(",", "_")
metadata.head()

In [ ]:
metadata.index=np.array(metadata["rna.bc"])
inter=intersection([rna.obs.index, metadata.index])
metadata=metadata.loc[inter]
rna=rna[inter]

In [ ]:
rna.obs.head()

In [ ]:
metadata.head()

In [ ]:
rna.obs=metadata.loc[rna.obs.index]
rna.obs.index=list(rna.obs["atac.bc"])
rna.obs.rename({"atac.bc" : "atac_bc", "rna.bc" : "rna_bc", "celltype" : "CellType"}, axis=1, inplace=True)
rna

In [ ]:
rna.write("Datasets/MouseSkin/CM/MouseSkin_GEX_Raw.h5ad", compression="gzip")

## mRNA

In [ ]:
adata=sc.read_h5ad("Datasets/MouseSkin/CM/MouseSkin_GEX_Raw.h5ad")
adata

In [ ]:
names = ["chr", "source", "type", "start", "stop", "score", "strand", "frame", "attribute"]
gtf_file="../AnnotRef/mm/gencode.vM33.annotation.gtf.gz"
gtf=pd.read_csv(gtf_file, sep="\t", header=None, comment="#", names=names, dtype={"chr": str})
gtf=gtf[(gtf["type"]=="gene") & (gtf["source"]=="HAVANA")]
gene_names=[attr.replace("gene_id", "gene_name").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_name")]
gene_names=[g.split(" ")[1][1:] for g in gene_names]
tmp = [attr.replace("gene_type", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_type ")]
if not tmp:
    tmp = [attr.replace("gene_biotype", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_biotype ")]

info=pd.DataFrame(data=[gene_names,tmp], index=["GeneName","GeneType"]).T

mRNA=adata[:, intersection([adata.var.index, info[info["GeneType"]=="protein_coding"]["GeneName"]])]
mRNA.X=scipy.sparse.csr_matrix(mRNA.X.copy())
mRNA.layers["raw"]=scipy.sparse.csr_matrix(mRNA.X.copy())
mRNA

In [ ]:
mRNA.write("Datasets/MouseSkin/CM/MouseSkin_mRNA_Raw.h5ad", compression="gzip")

## lncRNA

In [ ]:
names = ["chr", "source", "type", "start", "stop", "score", "strand", "frame", "attribute"]
gtf_file="../AnnotRef/mm/gencode.vM33.annotation.gtf.gz"
gtf=pd.read_csv(gtf_file, sep="\t", header=None, comment="#", names=names, dtype={"chr": str})
gtf=gtf[(gtf["type"]=="gene") & (gtf["source"]=="HAVANA")]
gene_names=[attr.replace("gene_id", "gene_name").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_name")]
gene_names=[g.split(" ")[1][1:] for g in gene_names]
tmp = [attr.replace("gene_type", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_type ")]
if not tmp:
    tmp = [attr.replace("gene_biotype", "").strip().strip("\"") for feature_attr in gtf.attribute for attr in feature_attr.split(";") if attr.strip().startswith("gene_biotype ")]

info=pd.DataFrame(data=[gene_names,tmp], index=["GeneName","GeneType"]).T

lncRNA=adata[:, intersection([adata.var.index, info[info["GeneType"]=="lncRNA"]["GeneName"]])]
lncRNA.X=scipy.sparse.csr_matrix(lncRNA.X.copy())
lncRNA.layers["raw"]=scipy.sparse.csr_matrix(lncRNA.X.copy())
lncRNA

In [ ]:
lncRNA.write("Datasets/MouseSkin/CM/MouseSkin_lncRNA_Raw.h5ad", compression="gzip")

# Processing

## Peak

In [ ]:
adata=sc.read_h5ad("Datasets/MouseSkin/CM/MouseSkin_Peak_Raw.h5ad")
adata.layers["raw"]=scipy.sparse.csr_matrix(adata.X.copy())
adata

In [ ]:
epi.pp.qc_stats(adata, verbose=False)

In [ ]:
min_features = 10**np.quantile(adata.obs["log_n_features"], 0.1)
max_features = 10**np.quantile(adata.obs["log_n_features"], 0.9)
min_cells = 10**np.quantile(adata.var["log_n_cells"], 0.05)
max_cells = 10**np.quantile(adata.var["log_n_cells"], 0.95)

In [ ]:
epi.pl.histogram(adata, "n_features", min_threshold=min_features, max_threshold=max_features, show_log=True)

In [ ]:
epi.pl.histogram(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, show_log=True)

In [ ]:
epi.pp.set_filter(adata, "n_features", min_threshold=min_features, max_threshold=max_features, verbose=False)
epi.pp.set_filter(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, verbose=False)
adata = epi.pp.apply_filters(adata, verbose=False)
adata

In [ ]:
y_train, y_test = sklearn.model_selection.train_test_split(adata.obs.index,	stratify=adata.obs.celltype, 
                                                            random_state=1, train_size=0.33)
adata=adata[y_train]
adata

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3')

In [ ]:
min_var = np.quantile(adata.var.variances_norm, 0.9)
adata=adata[:, adata.var.variances_norm>min_var]
adata

In [ ]:
epi.pp.normalize_total(adata)
adata = adata[:, adata.X.max(axis=0)>0]
adata = adata[adata.X.max(axis=1)>0]
adata

In [ ]:
adata.write("Datasets/MouseSkin/CM/MouseSkin_Peak_Def.h5ad", compression="gzip")

## GEX

In [ ]:
adata=sc.read_h5ad("Datasets/MouseSkin/CM/MouseSkin_GEX_Raw.h5ad")
adata.layers["raw"]=scipy.sparse.csr_matrix(adata.X.copy())
adata

In [ ]:
sc.pp.scrublet(adata)

In [ ]:
epi.pp.qc_stats(adata, verbose=True)
mt_gene_mask = np.flatnonzero([gene.startswith(("MT","mt")) for gene in adata.var.index])
adata.obs['mt_frac'] = np.sum(adata[:, mt_gene_mask].X, axis=1)/adata.obs['n_counts']

In [ ]:
min_features = 10**np.quantile(adata.obs["log_n_features"], 0.05)
max_features = 10**np.quantile(adata.obs["log_n_features"], 0.925)
min_cells = 10**np.quantile(adata.var["log_n_cells"], 0.25)
max_cells = 10**np.quantile(adata.var["log_n_cells"], 0.9)

In [ ]:
epi.pl.histogram(adata, "n_features", min_threshold=min_features, max_threshold=max_features, show_log=True)

In [ ]:
epi.pl.histogram(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, show_log=True)

In [ ]:
epi.pl.histogram(adata, "mt_frac", max_threshold=0.05, show_log=False)

In [ ]:
epi.pp.set_filter(adata, "n_features", min_threshold=min_features, max_threshold=max_features, verbose=False)
epi.pp.set_filter(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, verbose=False)
adata = epi.pp.apply_filters(adata, verbose=False)
adata = adata[adata.obs["mt_frac"]<=0.05]
adata=adata[adata.obs.predicted_doublet==False]
adata

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3')
min_var = np.quantile(adata.var.variances_norm, 0.7)
adata=adata[:, adata.var.variances_norm>min_var]

In [ ]:
subset=list(sc.read_h5ad("Datasets/MouseSkin/CM/MouseSkin_Peak_Def.h5ad").obs.index)
adata=adata[intersection([subset, adata.obs.index])]
adata

In [ ]:
epi.pp.normalize_total(adata)
sc.pp.log1p(adata)

In [ ]:
adata = adata[:, adata.X.max(axis=0)>0]
adata = adata[adata.X.max(axis=1)>0]
adata

In [ ]:
adata.write("Datasets/MouseSkin/CM/MouseSkin_GEX_Def.h5ad", compression="gzip")

## mRNA

In [ ]:
adata=sc.read_h5ad("Datasets/MouseSkin/CM/MouseSkin_mRNA_Raw.h5ad")
adata.layers["raw"]=scipy.sparse.csr_matrix(adata.X.copy())
adata

In [ ]:
sc.pp.scrublet(adata)

In [ ]:
epi.pp.qc_stats(adata, verbose=True)
mt_gene_mask = np.flatnonzero([gene.startswith(("MT","mt")) for gene in adata.var.index])
adata.obs['mt_frac'] = np.sum(adata[:, mt_gene_mask].X, axis=1)/adata.obs['n_counts']

In [ ]:
min_features = 10**np.quantile(adata.obs["log_n_features"], 0.05)
max_features = 10**np.quantile(adata.obs["log_n_features"], 0.95)
min_cells = 10**np.quantile(adata.var["log_n_cells"], 0.2)
max_cells = 10**np.quantile(adata.var["log_n_cells"], 0.95)

In [ ]:
epi.pl.histogram(adata, "n_features", min_threshold=min_features, max_threshold=max_features, show_log=True)

In [ ]:
epi.pl.histogram(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, show_log=True)

In [ ]:
epi.pl.histogram(adata, "mt_frac", max_threshold=0.05, show_log=False)

In [ ]:
epi.pp.set_filter(adata, "n_features", min_threshold=min_features, max_threshold=max_features, verbose=False)
epi.pp.set_filter(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, verbose=False)
adata = epi.pp.apply_filters(adata, verbose=False)
adata = adata[adata.obs["mt_frac"]<=0.05]
adata=adata[adata.obs.predicted_doublet==False]
adata

In [ ]:
subset=list(sc.read_h5ad("Datasets/MouseSkin/CM/MouseSkin_Peak_Def.h5ad").obs.index)
adata=adata[intersection([subset, adata.obs.index])]
adata

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3')
min_var = np.quantile(adata.var.variances_norm, 0.6)
adata=adata[:, adata.var.variances_norm>min_var]

In [ ]:
adata=adata[:, adata.var.variances_norm>min_var]
epi.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata = adata[:, adata.X.max(axis=0)>0]
adata = adata[adata.X.max(axis=1)>0]
adata

In [ ]:
adata.write("Datasets/MouseSkin/CM/MouseSkin_mRNA_Def.h5ad", compression="gzip")

## lncRNA

In [ ]:
adata=sc.read_h5ad("Datasets/MouseSkin/CM/MouseSkin_lncRNA_Raw.h5ad")
adata.layers["raw"]=scipy.sparse.csr_matrix(adata.X.copy())
adata

In [ ]:
epi.pp.qc_stats(adata, verbose=True)

In [ ]:
min_features = 10**np.quantile(adata.obs["log_n_features"], 0.05)
max_features = 10**np.quantile(adata.obs["log_n_features"], 0.95)
min_cells = 10**np.quantile(adata.var["log_n_cells"], 0.15)
max_cells = 10**np.quantile(adata.var["log_n_cells"], 0.95)

In [ ]:
epi.pl.histogram(adata, "n_features", min_threshold=min_features, max_threshold=max_features, show_log=True)

In [ ]:
epi.pl.histogram(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, show_log=True)

In [ ]:
epi.pp.set_filter(adata, "n_features", min_threshold=min_features, max_threshold=max_features, verbose=False)
epi.pp.set_filter(adata, "n_cells", min_threshold=min_cells, max_threshold=max_cells, verbose=False)
adata = epi.pp.apply_filters(adata, verbose=False)
adata

In [ ]:
subset=list(sc.read_h5ad("Datasets/MouseSkin/CM/MouseSkin_Peak_Def.h5ad").obs.index)
adata=adata[intersection([subset, adata.obs.index])]
adata

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3')
epi.pp.normalize_total(adata)
sc.pp.log1p(adata)
adata = adata[:, adata.X.max(axis=0)>0]
adata = adata[adata.X.max(axis=1)>0]
adata

In [ ]:
adata.write("Datasets/MouseSkin/CM/MouseSkin_lncRNA_Def.h5ad", compression="gzip")

## Muon objects

In [ ]:
import muon as mu
peak=sc.read_h5ad(f"Datasets/MouseSkin/CM/MouseSkin_Peak_Def.h5ad")
gex=sc.read_h5ad(f"Datasets/MouseSkin/CM/MouseSkin_GEX_Def.h5ad")
mrna=sc.read_h5ad(f"Datasets/MouseSkin/CM/MouseSkin_mRNA_Def.h5ad")
lncrna=sc.read_h5ad(f"Datasets/MouseSkin/CM/MouseSkin_lncRNA_Def.h5ad")

#Peak + GEX
print("Peak + GEX")
inter=intersection([peak.obs.index, gex.obs.index])
mudata=mu.MuData({"Peak" : peak[inter], "GEX" : gex[inter]})
mudata.write_h5mu(f"Datasets/MouseSkin/CM/MouseSkin_Peak_GEX_Def.h5mu", compression="gzip")

#Peak + mRNA
print("Peak + mRNA")
inter=intersection([peak.obs.index, mrna.obs.index])
mudata=mu.MuData({"Peak" : peak[inter], "mRNA" : mrna[inter]})
mudata.write_h5mu(f"Datasets/MouseSkin/CM/MouseSkin_Peak_mRNA_Def.h5mu", compression="gzip")

#Peak + lncRNA
print("Peak + lncRNA")
inter=intersection([peak.obs.index, lncrna.obs.index])
mudata=mu.MuData({"Peak" : peak[inter], "lncRNA" : lncrna[inter]})
mudata.write_h5mu(f"Datasets/MouseSkin/CM/MouseSkin_Peak_lncRNA_Def.h5mu", compression="gzip")

#mRNA + lncRNA
print("mRNA + lncRNA")
inter=intersection([mrna.obs.index, lncrna.obs.index])
mudata=mu.MuData({"mRNA" : mrna[inter], "lncRNA" : lncrna[inter]})
mudata.write_h5mu(f"Datasets/MouseSkin/CM/MouseSkin_mRNA_lncRNA_Def.h5mu", compression="gzip")

#Peak + mRNA + lncRNA
print("Peak + mRNA + lncRNA")
inter=intersection([peak.obs.index, mrna.obs.index, lncrna.obs.index])
mudata=mu.MuData({"Peak" : peak[inter], "mRNA" : mrna[inter], "lncRNA" : lncrna[inter]})
mudata.write_h5mu(f"Datasets/MouseSkin/CM/MouseSkin_Peak_mRNA_lncRNA_Def.h5mu", compression="gzip")

# CTA

In [ ]:
df=sc.read_h5ad("Datasets/MouseSkin/CM/MouseSkin_Peak_Def.h5ad").obs
df.head()

In [ ]:
df["RawCelltype"]=df["CellType"]
df["RawCelltype"]=df["RawCelltype"].replace("alowCD34+ bulge", "CD34 bulge")
df["RawCelltype"]=df["RawCelltype"].replace("ahighCD34+ bulge", "CD34 bulge")
df["RawCelltype"]=df["RawCelltype"].replace("TAC-1", "TAC")
df["RawCelltype"]=df["RawCelltype"].replace("TAC-2", "TAC")
df[["CellType","RawCelltype"]].to_csv("Datasets/MouseSkin/MouseSkin_Metadata.tsv.gz", sep="\t", compression="gzip")